# Ders 04 - Araç Kullanımı Tasarım Deseni

Bu derste Microsoft Agent Framework (Python) kullanarak AI ajanları için **Araç Kullanımı** tasarım desenini öğreneceksiniz. Şunları kapsıyoruz:

- `@tool` dekoratörü ve türlendirilmiş parametrelerle fonksiyon araçlarını tanımlama
- Modelin her aracın ne yaptığını bilmesi için araç şemaları sağlama
- `approval_mode` ile araç yürütmesini kontrol etme
- Pydantic modelleri ve `response_format` aracılığıyla **yapılandırılmış çıktı** döndürme

Senaryo, destinasyonları araştırabilen, uygunluğu kontrol edebilen ve uçuş bilgilerini alabilen bir **seyahat rezervasyon ajansı**dır.


## Kurulum


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool Dekoratörü ile Araç Tanımlama

`@tool` dekoratörü sade bir Python fonksiyonunu bir araça dönüştürür, böylece bir ajan tarafından çağrılabilir.
Önemli noktalar:

- **Docstring**, modelin gördüğü araç açıklaması olur.
- **Tip açıklamaları** (`Annotated` ile açıklamalar dahil) araç şemasını tanımlar.
- `approval_mode`, çağrının yürütülmeden önce kullanıcının onay vermesi gerekip gerekmediğini kontrol eder.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Birden Çok Araçla Ajan Oluşturma

Modelin kullanıcının sorusunu cevaplamak için ihtiyaç duyduğu her aracı çağırabilmesi için tüm üç aracı da istemciye iletin.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Araçlarla Yapılandırılmış Çıktı

`response_format` Pydantic modeline ayarlandığında, ajan serbest biçimli metin yerine iyi tanımlanmış bir JSON nesnesi döndürmek zorunda kalır. Bu, alt süreçteki kodun sonucu programlı olarak işlemesi gerektiğinde faydalıdır.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Araç Onay Desenleri

`@tool` üzerindeki `approval_mode` parametresi, araç çağrılarının çalıştırılmadan önce insan onayı gerektirip gerektirmediğini kontrol eder:

| Mod | Davranış |
|---|---|
| `"never_require"` | Araç otomatik olarak çalışır — kullanıcı onayı gerekmez. |
| `"always_require"` | Her çağrı, çalıştırılmadan önce kullanıcı tarafından onaylanmalıdır. |

Yan etkisi olan araçlar (örneğin, uçuş rezervasyonu, kredi kartı tahsilatı) için `"always_require"` kullanın, böylece insan sürecin içinde kalır.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Özet

Bu derste nasıl yapacağınızı öğrendiniz:

1. Araç şeması olarak hizmet eden tipli parametreler ve doküman dizeleri ile `@tool` dekoratörü kullanarak **araçları tanımlamayı**.
2. Karmaşık sorguları yanıtlamak için ajanın araçları sırayla çağırabilmesi için **birden çok aracı bileştirmeyi**.
3. `response_format` olarak bir Pydantic modeli geçirerek **yapılandırılmış çıktı döndürmeyi**.
4. Hassas işlemler için insan denetimi sağlamak amacıyla `approval_mode` ile **araç onayını kontrol etmeyi**.

Bu kalıplar, dış sistemlerle güvenli şekilde etkileşime girebilen güvenilir, üretim hazır ajanlar oluşturmanın temelini oluşturur.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba göstermemize rağmen, otomatik çevirilerin hatalar veya yanlışlıklar içerebileceğini lütfen unutmayın. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucunda oluşabilecek yanlış anlamalar veya yanlış yorumlamalar nedeniyle sorumluluk kabul edilmez.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
